In [1]:
import torch

print(f"PyTorch 版本: {torch.__version__}") # 應該要 >= 2.6.0
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"啟動成功！")
    print(f"顯卡名稱: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 驅動上限: {torch.version.cuda}")

PyTorch 版本: 2.11.0+cu126
CUDA 是否可用: True
啟動成功！
顯卡名稱: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA 驅動上限: 12.6


In [ ]:
import json
import chromadb
from chromadb.utils import embedding_functions

# 1. 讀取 JSON 檔案
with open("final_chunks.json", "r", encoding="utf-8") as f:
    final_chunks = json.load(f)

print(f"成功讀取 {len(final_chunks)} 個切片。")

# 2. 設定 Embedding 模型 (BGE-M3)
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"  # 有 NVIDIA 顯卡用 cuda，沒有則改 "cpu"
)

# 3. 初始化本地向量資料庫 (ChromaDB)，當前目錄建立一個名為 hiwin_vector_db 的資料夾
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 4. 建立或取得 Collection
collection = client.get_or_create_collection(
    name = "hiwin_manual",
    embedding_function = emb_fn,
    metadata = {"hnsw:space": "cosine"}
)

# 5. 執行 Embedding 並存入資料庫
documents = [c['content'] for c in final_chunks]
metadatas = [c['metadata'] for c in final_chunks]
ids = [f"id_{i}" for i in range(len(final_chunks))]

print(f"開始執行 BGE-M3 Embedding 轉換... (共 {len(documents)} 筆)")

# 分批寫入避免記憶體壓力
batch_size = 50
for i in range(0, len(documents), batch_size):
    end = i + batch_size
    collection.add(
        documents = documents[i:end],
        metadatas = metadatas[i:end],
        ids = ids[i:end]
    )
    print(f"已完成: {min(end, len(documents))}/{len(documents)}")

print(f"向量資料庫建置完成！資料夾路徑：./hiwin_vector_db")

成功讀取 87 個切片。


c:\Users\e11338\Desktop\Feed System GAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<?, ?it/s]


開始執行 BGE-M3 Embedding 轉換... (共 87 筆)
已完成: 50/87
已完成: 87/87
向量資料庫建置完成！資料夾路徑：./hiwin_vector_db
